# DiffInfinite - NIPA 병리 이미지 학습

NIPA 병리 패치 이미지(1024x1024, JPEG)를 이용한 DiffInfinite 모델 학습  
- 데이터: `../../data/NIPA/origin/` (클래스별 폴더)
- 모델 저장: `../../model/NIPA/diffinfinite/`

## 1. 환경 설정 및 Import

In [ ]:
import os
import sys
import math
import random
from pathlib import Path
from glob import glob

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, lr_scheduler
import torchvision.transforms as T
from torchvision import utils as vutils
from PIL import Image
from tqdm import tqdm
from ema_pytorch import EMA
from accelerate import Accelerator
from diffusers import DiffusionPipeline

sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from dm import Unet, GaussianDiffusion
from utils.helpers import exists, cycle, has_int_squareroot
from dataset import ComposeState, RandomRotate90

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "CPU only")

## 2. 설정값 정의

In [ ]:
# 경로
DATA_ROOT = "../../data/NIPA/origin"
MODEL_SAVE_DIR = "../../model/NIPA/diffinfinite"
RESULTS_DIR = os.path.join(MODEL_SAVE_DIR, "results")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# 모델 하이퍼파라미터
IMAGE_SIZE = 512          # 1024 -> 512 랜덤 크롭
Z_SIZE = IMAGE_SIZE // 8  # VAE latent size = 64
DIM = 256
DIM_MULTS = (1, 2, 4)
CHANNELS = 4              # VAE latent channels
RESNET_BLOCK_GROUPS = 2
BLOCK_PER_LAYER = 2
TIMESTEPS = 1000
SAMPLING_TIMESTEPS = 250

# 학습 하이퍼파라미터
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
TRAIN_NUM_STEPS = 250000
SAVE_SAMPLE_EVERY = 5000
SAVE_LOSS_EVERY = 100
GRADIENT_ACCUMULATE_EVERY = 2
NUM_SAMPLES = 4
NUM_WORKERS = 4
TRAIN_RATIO = 0.9

# 클래스 정보
CLASS_NAMES = sorted(os.listdir(DATA_ROOT))
NUM_CLASSES = len(CLASS_NAMES) + 1  # +1 for unconditional (class 0)
CLASS_TO_IDX = {name: idx + 1 for idx, name in enumerate(CLASS_NAMES)}

print(f"클래스 수: {NUM_CLASSES} (unconditional 포함)")
print(f"클래스 목록: {CLASS_TO_IDX}")
print(f"Latent size: {Z_SIZE}x{Z_SIZE}")

## 3. 데이터셋 정의

클래스별 폴더 구조에서 이미지를 로드하고, DiffInfinite가 요구하는 형식(이미지 + 마스크)으로 변환합니다.  
- 세그멘테이션 마스크가 없으므로 클래스 레이블로 채운 uniform mask를 생성합니다.
- 1024x1024 이미지에서 512x512 랜덤 크롭을 수행합니다.